# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Reading from Silver Table

In [0]:
df = spark.table("workspace.silver.crm.customers")

# The Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_key, -- Surrogate key
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_line,
    pn.start_date
FROM silver.crm_products pn
LEFT JOIN silver.erp_product_category pc
    ON pn.category_id = pc.category_id
--WHERE pn.end_date IS NULL; -- Filter out all historical data
"""
df = spark.sql(query)


In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")

## Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_products LIMIT 10